# CSC 303 – Academic Advisor System
Model training notebook (run on Google Colab). Run each cell from top to bottom; the last cell downloads `advisor_model.pkl`.

### Step 1 - Imports

In [ ]:
!pip install scikit-learn==1.9.1

In [2]:
import pickle
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("scikit-learn version:", sklearn.__version__)  # write this down - use the same version in VS Code


scikit-learn version: 1.9.1


### Step 2 - Create the dataset (synthetic student records)

In [3]:
# Each row is one student. We simulate realistic study habits and compute
# the academic standing they are likely to reach on the 5.0 CGPA scale.
rng = np.random.default_rng(42)
N = 2000

df = pd.DataFrame({
    "current_cgpa":          rng.uniform(1.0, 5.0, N).round(2),
    "study_hours":           rng.integers(0, 41, N),        # hours per week
    "attendance":            rng.integers(30, 101, N),      # % of classes attended
    "assignment_completion": rng.integers(20, 101, N),      # % of assignments submitted
    "course_units":          rng.integers(12, 31, N),       # units this semester
    "sleep_hours":           rng.uniform(3, 10, N).round(1),# hours per night
    "extracurricular_hours": rng.integers(0, 21, N),        # hours per week
})

# Projected GPA = past performance + effect of habits + random noise
projected = (
    0.65 * df["current_cgpa"]
    + 0.025 * df["study_hours"]
    + 0.008 * (df["attendance"] - 70)
    + 0.006 * (df["assignment_completion"] - 70)
    - 0.02 * (df["course_units"] - 21)
    - 0.08 * np.abs(df["sleep_hours"] - 7.5)
    - 0.01 * np.clip(df["extracurricular_hours"] - 10, 0, None)
    + 0.9
    + rng.normal(0, 0.15, N)
).clip(0, 5)

def standing(gpa):
    if gpa >= 4.50: return "First Class"
    if gpa >= 3.50: return "Second Class Upper"
    if gpa >= 2.40: return "Second Class Lower"
    return "Third Class / At Risk"

df["standing"] = projected.apply(standing)
print(df.head())
print(df["standing"].value_counts())


   current_cgpa  study_hours  attendance  assignment_completion  course_units  \
0          4.10           13          73                     72            18   
1          2.76           34          88                     78            16   
2          4.43            1          75                     41            21   
3          3.79           18          72                     48            24   
4          1.38           32          63                     96            16   

   sleep_hours  extracurricular_hours            standing  
0          6.2                     20  Second Class Upper  
1          3.8                     11  Second Class Upper  
2          7.0                      0  Second Class Upper  
3          4.1                     18  Second Class Lower  
4          3.4                     19  Second Class Lower  
standing
Second Class Lower       826
Second Class Upper       586
Third Class / At Risk    504
First Class               84
Name: count, dtype: int64


### Step 3 - Split into features (X) and label (y), then train/test

In [4]:
FEATURES = ["current_cgpa", "study_hours", "attendance", "assignment_completion",
            "course_units", "sleep_hours", "extracurricular_hours"]
X = df[FEATURES]
y = df["standing"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)


Train: (1600, 7)  Test: (400, 7)


### Step 4 - Build and train the model

In [5]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)),
])
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](4,)","['First Class','Second Class Lower','Second Class Upper', 'Third Class / At Risk']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['current_cgpa','study_hours','attendance',...,'course_units', 'sleep_hours','extracurricular_hours']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


### Step 5 - Evaluate

In [6]:
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

importances = pd.Series(model.named_steps["classifier"].feature_importances_, index=FEATURES)
print(importances.sort_values(ascending=False))


Accuracy: 83.5 %
                       precision    recall  f1-score   support

          First Class       1.00      0.24      0.38        17
   Second Class Lower       0.80      0.90      0.85       165
   Second Class Upper       0.83      0.86      0.85       117
Third Class / At Risk       0.90      0.79      0.84       101

             accuracy                           0.83       400
            macro avg       0.88      0.70      0.73       400
         weighted avg       0.84      0.83      0.83       400

[[  4   0  13   0]
 [  0 149   7   9]
 [  0  16 101   0]
 [  0  21   0  80]]
current_cgpa             0.531888
study_hours              0.132151
attendance               0.081925
assignment_completion    0.078235
sleep_hours              0.068292
course_units             0.059614
extracurricular_hours    0.047895
dtype: float64


### Step 6 - Test with one student (yourself!)

In [7]:
me = pd.DataFrame([{
    "current_cgpa": 3.8, "study_hours": 15, "attendance": 85, "assignment_completion": 90,
    "course_units": 24, "sleep_hours": 6.5, "extracurricular_hours": 5,
}])
print("Predicted standing:", model.predict(me)[0])


Predicted standing: Second Class Upper


### Step 7 - Save the model as a .pkl file and download it

In [ ]:
with open("advisor_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Saved advisor_model.pkl")

try:
    from google.colab import files  # only works inside Colab
    files.download("advisor_model.pkl")
except ImportError:
    pass


Saved advisor_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>